# Sunflower 14B GRPO Combined — Inference Notebook

This notebook runs inference on the `jq/sunflower-14b-bs64-lr1e-4-250919` base model combined with the `jq/sunflower-14b-grpo-combined` LoRA adapter, using **Unsloth + vLLM** for fast generation. It demonstrates:

1. Environment and dependency setup
2. Loading the base model and LoRA adapter
3. Running translation prompts across Ugandan languages (Luganda, Runyankole, Lusoga, Acholi, Ateso)
4. Running general-knowledge and reasoning prompts

## Setup / Dependency Installation

The commented commands below reinstall PyTorch, Transformers, vLLM, Unsloth and related CUDA 13.0 wheels. Uncomment and run once per fresh environment.

In [ ]:
# !uv pip install --force-reinstall --no-cache-dir \
#   torchaudio torchvision torchcodec \
#   --index-url https://download.pytorch.org/whl/cu130

# !uv pip install --force-reinstall transformers torch accelerate vllm unsloth huggingface_hub bitsandbytes "xformers==0.0.33.post1" --torch-backend=cu130

# uv pip install --force-reinstall https://github.com/vllm-project/vllm/releases/download/v0.17.1/vllm-0.17.1+cu130-cp38-abi3-manylinux_2_35_x86_64.whl

## Imports and Environment Flags

- `FastLanguageModel` (Unsloth) loads the base model with vLLM fast-inference enabled.
- `LLM` / `SamplingParams` / `LoRARequest` come from vLLM for generation and LoRA hot-swapping.
- `UNSLOTH_VLLM_STANDBY=0` avoids a known crash when standby mode is enabled.
- `FLASHINFER_DISABLE_VERSION_CHECK=1` skips a strict FlashInfer version check.

In [ ]:
import os
from unsloth import FastLanguageModel
import torch
from huggingface_hub import snapshot_download, login
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

os.environ["UNSLOTH_VLLM_STANDBY"] = "0" # Causes crashes when set to 1
os.environ["FLASHINFER_DISABLE_VERSION_CHECK"] ="1"

## Authenticate with Hugging Face

Required because the base model and adapter repositories are gated. Prompts for an access token interactively.

In [ ]:
login()

## Model Paths and System Prompt

- `BASE_MODEL_PATH` — the SFT base checkpoint.
- `GRPO_LORA_PATH` — the GRPO-trained LoRA adapter merged at inference time.
- `SYSTEM_MESSAGE` — the Sunflower persona prompt prepended to every chat request.

In [ ]:
BASE_MODEL_PATH = "jq/sunflower-14b-bs64-lr1e-4-250919"
GRPO_LORA_PATH = "jq/sunflower-14b-grpo-combined"
SYSTEM_MESSAGE = """You are Sunflower, a helpful assistant made by Sunbird AI who understands all Ugandan languages. You specialise in accurate translations, 
explanations, summaries and other language tasks."""

## Download the GRPO LoRA Adapter

`snapshot_download` pulls the full adapter repository (weights + `adapter_config.json`) to a local cache directory. The resulting path is passed to `LoRARequest` later.

In [ ]:
model_lora_path = snapshot_download(repo_id=GRPO_LORA_PATH)

## Inference with Base Model

In [ ]:
max_seq_length = 1024
lora_rank = 32 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL_PATH,
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)


### Example instruction (English → Luganda translation)

In [ ]:
instruction  = "Translate to luganda: I am watching an Arsenal game right now"

### Build the chat prompt and sampling configuration

The tokenizer's chat template formats the system + user messages into the model's expected prompt format. `SamplingParams` controls decoding: temperature 0.6, top-k 50, up to 1024 generated tokens.

In [ ]:
messages = [
    {"role": "system", "content": SYSTEM_MESSAGE},
    {"role": "user",   "content": instruction},
]

prompt = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
sampling_params = SamplingParams(
    temperature = 0.6,
    top_k = 50,
    max_tokens = 1024,
)


### Run generation with the LoRA adapter attached

`fast_generate` routes the request through vLLM. `LoRARequest` tells vLLM to dynamically load and apply the downloaded adapter for this call — the base weights stay untouched, so adapters can be swapped per request.

In [ ]:
output = model.fast_generate(
    [prompt],
    sampling_params = sampling_params,
    lora_request = LoRARequest(
            lora_name="adapter",   # any unique name/identifier
            lora_int_id=1,            # unique integer ID
            lora_path=model_lora_path,  # local dir with adapter_config.json etc.
        ),
)



### Inspect raw output and extract the generated text

`output` is a list of vLLM `RequestOutput` objects; `output[0].outputs[0].text` is the decoded completion string.

In [ ]:
output

In [ ]:
response = output[0].outputs[0].text

In [ ]:
response

## Inference with Lora Adapter using vllm

In [ ]:
def generate_response(instruction, temperature=0.6):
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user",   "content": instruction},
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True, # Must add for generation
        tokenize = False,
    )
    sampling_params = SamplingParams(
        temperature = temperature,
        top_k = 50,
        max_tokens = 1024,
    )
    output = model.fast_generate(
        [prompt],
        sampling_params = sampling_params,
        lora_request = LoRARequest(
                lora_name="adapter",   # any unique name/identifier
                lora_int_id=1,            # unique integer ID
                lora_path=model_lora_path,  # local dir with adapter_config.json etc.
            ),
    )
    response = output[0].outputs[0].text
    return response

### Helper: `generate_response(instruction)`

Wraps prompt formatting, sampling, and LoRA-enabled vLLM generation into a single reusable function used by the test cells below.

### Translation tests across Ugandan languages
The following cells translate the same English sentence into Luganda, Runyankole, Lusoga, Acholi, and Ateso to spot-check cross-lingual quality.

In [ ]:
instruction = "Translate to luganda: The winner of Sunday's Carabao Cup Final will gain qualification to the Europa Conference League play-off round next season."
res = generate_response(instruction, temperature=0.1)
res

In [ ]:
instruction = "Translate to runyakole: The winner of Sunday's Carabao Cup Final will gain qualification to the Europa Conference League play-off round next season."
res = generate_response(instruction, temperature=0.1)
res

In [ ]:
instruction = "Translate to lusoga: The winner of Sunday's Carabao Cup Final will gain qualification to the Europa Conference League play-off round next season."
res = generate_response(instruction, temperature=0.3)
res

In [ ]:
instruction = "Translate to acholi: The winner of Sunday's Carabao Cup Final will gain qualification to the Europa Conference League play-off round next season."
res = generate_response(instruction, temperature=0.3)
res

In [ ]:
instruction = "Translate to ateso: The winner of Sunday's Carabao Cup Final will gain qualification to the Europa Conference League play-off round next season."
res = generate_response(instruction, temperature=0.1)
res

## General Knowledge Q&A

Non-translation prompts to probe the model's open-domain knowledge and its grounding in Sunbird AI / Sunflower identity.

In [ ]:
instruction = "Who is Sunbird AI, what do they do?"
res = generate_response(instruction)
res

In [ ]:
instruction = "What is Sunflower"
res = generate_response(instruction)
res

In [ ]:
from IPython.display import display, Markdown

### Pretty-print responses as Markdown

`display(Markdown(res))` renders formatting (lists, bold, etc.) inline for easier review of longer answers.

In [ ]:
display(Markdown(res))

In [ ]:
instruction = "Uganda yafuna ddi ebwetogoze?"
res = generate_response(instruction)
display(Markdown(res))

### Ugandan-language questions and long-form translations

Mixes Luganda-language questions (understanding + response in Luganda), knowledge cut-off probes, and longer English→Luganda passages to stress-test fluency on extended context.

In [ ]:
instruction = "Uganda yakabera neba pulezidenti bameka, era bebani?"
res = generate_response(instruction)
display(Markdown(res))

In [ ]:
instruction = "When is Luwum day is Uganda?"
res = generate_response(instruction)
display(Markdown(res))

In [ ]:
instruction = """Translate to Luganda: The English Football League Cup, often referred to as the League Cup and officially known as the Carabao Cup for sponsorship reasons,
is an annual knockout competition in men's domestic football in England"""
res = generate_response(instruction, temperature=0.1)
display(Markdown(res))

In [ ]:
instruction = """What is the current weather now?"""
res = generate_response(instruction)
display(Markdown(res))

In [ ]:
instruction = """What was the result from the Caraboa cup final of Arsenal Vs Manchester City?"""
res = generate_response(instruction)
display(Markdown(res))

In [ ]:
instruction = """When is Arsenal playing Sporting Lisbon in the Champions league"""
res = generate_response(instruction)
display(Markdown(res))